In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations
from datetime import datetime

In [2]:
# ref: https://www.kaggle.com/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data
data = pd.read_csv('../datasets/Amazon Sale Report.csv', parse_dates=['Date'])
data

C:\Users\rah\AppData\Local\Temp\ipykernel_9448\1751934001.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../datasets/Amazon Sale Report.csv', parse_dates=['Date'])
C:\Users\rah\AppData\Local\Temp\ipykernel_9448\1751934001.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv('../datasets/Amazon Sale Report.csv', parse_dates=['Date'])


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128970,128970,406-6001380-7673107,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,JNE3697,JNE3697-KR-XL,kurta,...,INR,517.00,HYDERABAD,TELANGANA,500013.0,IN,NaN,False,NaN,False
128971,128971,402-9551604-7544318,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,SET401,SET401-KR-NP-M,Set,...,INR,999.00,GURUGRAM,HARYANA,122004.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN,False
128972,128972,407-9547469-3152358,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,J0157,J0157-DR-XXL,Western Dress,...,INR,690.00,HYDERABAD,TELANGANA,500049.0,IN,NaN,False,NaN,False
128973,128973,402-6184140-0545956,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,J0012,J0012-SKD-XS,Set,...,INR,1199.00,Halol,Gujarat,389350.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,False,NaN,False


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   index               128975 non-null  int64         
 1   Order ID            128975 non-null  object        
 2   Date                128975 non-null  datetime64[ns]
 3   Status              128975 non-null  object        
 4   Fulfilment          128975 non-null  object        
 5   Sales Channel       128975 non-null  object        
 6   ship-service-level  128975 non-null  object        
 7   Style               128975 non-null  object        
 8   SKU                 128975 non-null  object        
 9   Category            128975 non-null  object        
 10  Size                128975 non-null  object        
 11  ASIN                128975 non-null  object        
 12  Courier Status      122103 non-null  object        
 13  Qty                 128975 no

# Step 1 — Data Cleaning & Validation

1. **Drop the `Unnamed: 22` column entirely**

2. **Convert `ship-postal-code` to string** (it's a code, not a number)

3. **Parse `Date`** into the following features:
   - `year`
   - `month`
   - `day`
   - `weekday`
   - `week-of-year`

4. **Identify orders where `Amount` is missing**
   - Quantify what percentage of **total orders** they represent
   - Quantify what percentage of **quantity** they represent

5. **Flag rows where `currency` is missing but `Amount` is present** (inconsistency)

6. **Detect duplicate `Order ID` values**
   - Investigate whether they are **true duplicates** or **multi-item orders**

In [4]:
# 1
msg = 'ℹ️ columns will be dropped' if 'Unnamed: 22' in data.columns else 'ℹ️ column does not exist'
print(msg)
data.drop(columns=['Unnamed: 22'], axis=1, inplace=True, errors='ignore')
# 2
data['ship-postal-code'] = data['ship-postal-code'].astype('string') # it is already 
# 3
def add_date_features(df: pd.DataFrame, date_column: str|datetime) -> pd.DataFrame:
    date_df = df.copy()
    date_df[date_column] = pd.to_datetime(date_df[date_column], errors="coerce")
    
    date_df["year"] = date_df[date_column].dt.year
    date_df["month"] = date_df[date_column].dt.month
    date_df["day"] = date_df[date_column].dt.day
    date_df["weekday"] = date_df[date_column].dt.weekday # 0=Monday, 6=Sunday
    date_df["weekday_name"] = date_df[date_column].dt.day_name()
    date_df["week_of_year"] = date_df[date_column].dt.isocalendar().week.astype("Int64")
    date_df["quarter"] = date_df[date_column].dt.quarter
    date_df["day_of_year"] = date_df[date_column].dt.dayofyear

    return date_df
data = add_date_features(data, date_column='Date')
# 4
pct_orders_missing =  data['Amount'].isna().sum() * 100.0 / len(data['Amount'])
pct_quantity_missing = data.loc[data["Amount"].isna(), "Qty"].sum() * 100.0 / data["Qty"].sum()
print(f"Missing Amount — % of orders:   {pct_orders_missing:.2f}%")
print(f"Missing Amount — % of quantity: {pct_quantity_missing:.2f}%")
# 5
data['has_missing_currency'] = (data['currency'].isna() & ~data['Amount'].isna())
# 6
n_unique_duplicated_ids  = (data["Order ID"].value_counts() > 1).sum()
n_extra_rows             = data["Order ID"].duplicated().sum()
n_all_rows_in_dups       = data["Order ID"].duplicated(keep=False).sum()
n_unique_duplicated      = data[data["Order ID"].duplicated(keep=False)]["Order ID"].nunique()

print(f"Unique duplicated Order IDs:        {n_unique_duplicated_ids}")
print(f"Extra rows (beyond first):          {n_extra_rows}")
print(f"All rows that belong to a dup ID:   {n_all_rows_in_dups}")
print(f"Unique duplicated Order IDs:        {n_unique_duplicated}")
print(f"Unique duplicated Order IDs:        {n_all_rows_in_dups - n_extra_rows}")

ℹ️ columns will be dropped
Missing Amount — % of orders:   6.04%
Missing Amount — % of quantity: 0.14%
Unique duplicated Order IDs:        6846
Extra rows (beyond first):          8597
All rows that belong to a dup ID:   15443
Unique duplicated Order IDs:        6846
Unique duplicated Order IDs:        6846


# Step 2 — Order Status Classification

The `Status` column has many values (Cancelled, Shipped, Shipped - Delivered to Buyer, Shipped - Returned to Seller, etc.)

## 1. Create `Order_Outcome` column

Map raw statuses into clean categories:

| Category |
|---|
| Delivered |
| In-Transit |
| Cancelled |
| Returned |
| Pending |
| Other |

## 2. Create binary `Is_Lost_Revenue` column

```
Is_Lost_Revenue = 1  if the order is Cancelled or Returned
Is_Lost_Revenue = 0  otherwise
```

In [23]:
STATUS_CATEGORY_MAP = {
    "Shipped - Delivered to Buyer": "Delivered",

    "Shipped": "In-Transit",
    "Shipped - Out for Delivery": "In-Transit",
    "Shipped - Picked Up": "In-Transit",
    "Shipped - Lost in Transit": "In-Transit",
    "Shipped - Returning to Seller": "In-Transit",
    "Shipping": "In-Transit",

    "Cancelled": "Cancelled",
    "Shipped - Rejected by Buyer": "Cancelled",

    "Pending": "Pending",
    "Pending - Waiting for Pick Up": "Pending",

    "Shipped - Returned to Seller": "Returned",

    "Shipped - Damaged": "Other",
}

data['order_outcome'] = data['Status'].map(STATUS_CATEGORY_MAP).fillna('Other').astype('category')
data['is_lost_revenue'] = data['order_outcome'].isin(['Cancelled', 'Returned']).astype('int8')
data

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,month,day,weekday,weekday_name,week_of_year,quarter,day_of_year,has_missing_currency,order_outcome,is_list_revenue
0,0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,4,30,5,Saturday,17,2,120,False,Cancelled,1
1,1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,4,30,5,Saturday,17,2,120,False,Delivered,0
2,2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,4,30,5,Saturday,17,2,120,False,In-Transit,0
3,3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,4,30,5,Saturday,17,2,120,False,Cancelled,1
4,4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,4,30,5,Saturday,17,2,120,False,In-Transit,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128970,128970,406-6001380-7673107,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,JNE3697,JNE3697-KR-XL,kurta,...,5,31,1,Tuesday,22,2,151,False,In-Transit,0
128971,128971,402-9551604-7544318,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,SET401,SET401-KR-NP-M,Set,...,5,31,1,Tuesday,22,2,151,False,In-Transit,0
128972,128972,407-9547469-3152358,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,J0157,J0157-DR-XXL,Western Dress,...,5,31,1,Tuesday,22,2,151,False,In-Transit,0
128973,128973,402-6184140-0545956,2022-05-31,Shipped,Amazon,Amazon.in,Expedited,J0012,J0012-SKD-XS,Set,...,5,31,1,Tuesday,22,2,151,False,In-Transit,0
